# 03 — Group-Aware Stratified Split & Zero-Leakage Verification
## Plant Disease Classification Using Vision Transformers

### Objectives:
1. **Group-Aware Stratified Split**: Split the dataset into 70% Train, 15% Validation, and 15% Test while preserving the class distribution.
2. **Cluster Isolation**: Ensure all images within the same near-duplicate cluster (pHash group) are assigned to the exact same split to prevent data leakage.
3. **Cross-Verification**: Perform strict verification to guarantee Zero Leakage across the Train, Validation, and Test sets.
4. **Portable Export**: Save the split records using relative paths (`rel_path`) to ensure seamless training execution across both Local and Google Colab environments.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

sys.path.append(os.path.abspath(".."))

In [12]:

import hashlib
from pathlib import Path

import imagehash
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split


# 1. Setup paths
data_raw_root = Path("../data/raw")
processed_dir = Path("../data/processed")
split_dir = processed_dir / "split"

split_dir.mkdir(parents=True, exist_ok=True)

valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Locate color directory
color_candidates = [
    p for p in data_raw_root.rglob("color")
    if p.is_dir()
]

if not color_candidates:
    raise FileNotFoundError(
        f"Could not find a 'color' directory inside {data_raw_root}"
    )

color_dir = color_candidates[0]
print(f"Targeting dataset: {color_dir}")


# 2. Hash utilities
def calculate_md5(path: Path) -> str:
    """Calculate MD5 from file bytes."""
    md5 = hashlib.md5()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            md5.update(chunk)

    return md5.hexdigest()


def calculate_phash(path: Path) -> str:
    """Calculate perceptual hash from image content."""
    with Image.open(path) as image:
        return str(imagehash.phash(image.convert("RGB")))


# 3. Scan images and build metadata
all_images = sorted(
    p for p in color_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in valid_exts
)

if not all_images:
    raise ValueError(f"No images found inside {color_dir}")

records = []

for path in all_images:
    # Assumes the immediate parent folder is the class label.
    label = path.parent.name

    try:
        md5 = calculate_md5(path)
        phash = calculate_phash(path)
    except Exception as exc:
        print(f"Skipping unreadable image {path}: {exc}")
        continue

    records.append({
        "rel_path": path.relative_to(color_dir).as_posix(),
        "label": label,
        "plant": "rice",
        "condition": label,
        "md5": md5,
        "phash": phash,
    })

df_all = pd.DataFrame(records)

if df_all.empty:
    raise ValueError("No readable images found.")

# Stable class IDs
classes = sorted(df_all["label"].unique())
class_to_id = {
    class_name: idx
    for idx, class_name in enumerate(classes)
}
df_all["class_id"] = df_all["label"].map(class_to_id).astype(int)

# Exact pHash grouping.
# Images with identical pHash are assigned to the same group.
df_all["phash_group"] = "phash_" + df_all["phash"]

# Also group byte-identical files together.
# This catches exact duplicates even if metadata or image decoding differs.
df_all["md5_group"] = "md5_" + df_all["md5"]

# Combine duplicate relationships using connected components.
# If two files share either MD5 or pHash, they belong to the same group.
parent = list(range(len(df_all)))


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    root_a = find(a)
    root_b = find(b)
    if root_a != root_b:
        parent[root_b] = root_a


for column in ("md5_group", "phash_group"):
    first_seen = {}

    for idx, value in enumerate(df_all[column]):
        if value in first_seen:
            union(idx, first_seen[value])
        else:
            first_seen[value] = idx

df_all["group_root"] = [find(i) for i in range(len(df_all))]
df_all["phash_group"] = (
    "group_" + df_all["group_root"].astype(str)
)

# Remove internal helper columns from final exports later.
print(f"Readable images: {len(df_all):,}")
print(f"Unique groups: {df_all['phash_group'].nunique():,}")


# 4. Build one record per group for splitting
# A group containing multiple labels must be reviewed.
label_counts = df_all.groupby("phash_group")["label"].nunique()
conflicting_groups = label_counts[label_counts > 1]

if not conflicting_groups.empty:
    raise ValueError(
        f"{len(conflicting_groups)} duplicate groups contain "
        "multiple labels. Review these groups before splitting."
    )

df_groups = (
    df_all.groupby("phash_group", as_index=False)
    .first()
)

# Check if stratified splitting is feasible
class_group_counts = df_groups["class_id"].value_counts()

if (class_group_counts < 3).any():
    raise ValueError(
        "At least one class has fewer than 3 independent groups. "
        "Stratified train/val/test splitting is not feasible."
    )


# 5. Stratified group split: 70% / 15% / 15%
train_groups, temp_groups = train_test_split(
    df_groups,
    test_size=0.30,
    stratify=df_groups["class_id"],
    random_state=42,
)

val_groups, test_groups = train_test_split(
    temp_groups,
    test_size=0.50,
    stratify=temp_groups["class_id"],
    random_state=42,
)


# 6. Assign every image according to its group
train_ids = set(train_groups["phash_group"])
val_ids = set(val_groups["phash_group"])
test_ids = set(test_groups["phash_group"])

df_train = df_all[df_all["phash_group"].isin(train_ids)].copy()
df_val = df_all[df_all["phash_group"].isin(val_ids)].copy()
df_test = df_all[df_all["phash_group"].isin(test_ids)].copy()


# 7. Export exactly the requested fields
export_cols = [
    "rel_path",
    "label",
    "class_id",
    "plant",
    "condition",
    "md5",
    "phash",
    "phash_group",
]

for name, df in [
    ("train", df_train),
    ("val", df_val),
    ("test", df_test),
]:
    df[export_cols].to_csv(
        split_dir / f"{name}.csv",
        index=False,
    )


# 8. Verify no group leakage
assert set(df_train["phash_group"]).isdisjoint(
    set(df_val["phash_group"])
)
assert set(df_train["phash_group"]).isdisjoint(
    set(df_test["phash_group"])
)
assert set(df_val["phash_group"]).isdisjoint(
    set(df_test["phash_group"])
)

print("\n" + "=" * 45)
print("SPLIT SUMMARY")
print("=" * 45)

for name, df in [
    ("Train", df_train),
    ("Validation", df_val),
    ("Test", df_test),
]:
    print(
        f"{name:10}: {len(df):,} images "
        f"({len(df) / len(df_all) * 100:.1f}%), "
        f"{df['phash_group'].nunique():,} groups"
    )

print(f"\nCSV files exported to: {split_dir}")
print(f"Export columns: {export_cols}")


Targeting dataset: ../data/raw/plantvillage dataset/color
Readable images: 54,305
Unique groups: 54,283

SPLIT SUMMARY
Train     : 38,011 images (70.0%), 37,998 groups
Validation: 8,147 images (15.0%), 8,142 groups
Test      : 8,147 images (15.0%), 8,143 groups

CSV files exported to: ../data/processed/split
Export columns: ['rel_path', 'label', 'class_id', 'plant', 'condition', 'md5', 'phash', 'phash_group']


## 2. Check Relative Path

In [15]:
train_df = pd.read_csv("../data/processed/split/train.csv")
val_df = pd.read_csv("../data/processed/split/val.csv")
test_df = pd.read_csv("../data/processed/split/test.csv")

print("Data sample from train.csv:")
print(train_df[['rel_path', 'class_id', 'class_id']].head())


Data sample from train.csv:
                                            rel_path  class_id  class_id
0  Apple___Apple_scab/00075aa8-d81a-4184-8541-b69...         0         0
1  Apple___Apple_scab/0208f4eb-45a4-4399-904e-989...         0         0
2  Apple___Apple_scab/023123cb-7b69-4c9f-a521-766...         0         0
3  Apple___Apple_scab/029424b0-0ef5-491b-9ef5-069...         0         0
4  Apple___Apple_scab/03354abb-aa1c-4f9d-a1ef-9f4...         0         0


## 3. Zero Leakage Verification


In [16]:
# 1. Collect group IDs and MD5 hashes for each split
train_groups = set(train_df["phash_group"].dropna())
val_groups = set(val_df["phash_group"].dropna())
test_groups = set(test_df["phash_group"].dropna())

train_md5s = set(train_df["md5"].dropna())
val_md5s = set(val_df["md5"].dropna())
test_md5s = set(test_df["md5"].dropna())


# 2. Check perceptual-hash group overlap
overlap_train_val = train_groups & val_groups
overlap_train_test = train_groups & test_groups
overlap_val_test = val_groups & test_groups


# 3. Check exact duplicate overlap using MD5
overlap_train_val_md5 = train_md5s & val_md5s
overlap_train_test_md5 = train_md5s & test_md5s
overlap_val_test_md5 = val_md5s & test_md5s


# 4. Print validation report
print("=" * 60)
print("DATA LEAKAGE VALIDATION REPORT")
print("=" * 60)

print("\n[Perceptual Hash Group Overlap]")
print(f"Train-Val:  {len(overlap_train_val)}")
print(f"Train-Test: {len(overlap_train_test)}")
print(f"Val-Test:   {len(overlap_val_test)}")

print("\n[Exact Duplicate (MD5) Overlap]")
print(f"Train-Val:  {len(overlap_train_val_md5)}")
print(f"Train-Test: {len(overlap_train_test_md5)}")
print(f"Val-Test:   {len(overlap_val_test_md5)}")


# 5. Determine validation results
phash_leakage = any([
    overlap_train_val,
    overlap_train_test,
    overlap_val_test,
])

md5_leakage = any([
    overlap_train_val_md5,
    overlap_train_test_md5,
    overlap_val_test_md5,
])

print("\n" + "=" * 60)
print("VALIDATION RESULT")
print("=" * 60)

if phash_leakage or md5_leakage:
    print("FAILED: Potential data leakage detected.")

    if phash_leakage:
        print("- Perceptual-hash groups overlap across splits.")

    if md5_leakage:
        print("- Exact duplicate files overlap across splits.")

    raise AssertionError(
        "Data leakage detected. Review the splitting procedure."
    )

print("PASSED: No cross-split overlap detected.")
print("- No shared perceptual-hash groups.")
print("- No shared exact MD5 hashes.")
print("=" * 60)

DATA LEAKAGE VALIDATION REPORT

[Perceptual Hash Group Overlap]
Train-Val:  0
Train-Test: 0
Val-Test:   0

[Exact Duplicate (MD5) Overlap]
Train-Val:  0
Train-Test: 0
Val-Test:   0

VALIDATION RESULT
PASSED: No cross-split overlap detected.
- No shared perceptual-hash groups.
- No shared exact MD5 hashes.
